# **1. Opportunity signals**

This notebook builds an exploratory recruitment layer for identifying players aged 16–23 whose playing-time sequence is unusual for their age and competitive context.

The signal measures **revealed opportunity or selection propensity**, not talent. The early watch identifies the first unusually rapid opportunity accumulation in each player's observed career.

## **2. Statistical design**

1. `played`, `in squad`, and `not in squad` define the observed selection-decision universe.
2. `injured`, `absent`, and `played for a different team` are not treated as zero opportunity.
3. Each league–season–round reference is calculated with players of **all ages**.
4. The 16–23 age restriction is applied only after this normalization.
5. A descriptive opportunity residual is calculated within league, season, and integer age.
6. EWMA accumulates persistence continuously across each player's observed career; club or league changes do not reset it.
7. A player's first and only watch starts when two observations in a rolling window of up to three eligible decisions have age-adjusted residuals of at least +0.75 and cumulative minutes are positive.
8. Three decisions below the positional median close the episode and allow a later reactivation.

## **3. Data preparation and signal construction**

Stable player labels are derived from roster history, while unavailable states are excluded from the opportunity denominator. Minutes are normalized against the complete league–round population before age-specific residuals and sequential persistence measures are calculated.

In [1]:
from pathlib import Path
from collections import defaultdict
import json

import numpy as np
import pandas as pd

ROOT = Path.cwd().resolve()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
RAW = ROOT / 'data' / 'raw'
PROCESSED = ROOT / 'data' / 'processed'
ASSETS = ROOT / 'assets'
ASSETS.mkdir(parents=True, exist_ok=True)
VIZ_DIR = ROOT / 'viz'
VIZ_DIR.mkdir(parents=True, exist_ok=True)

LEAGUE_NAMES = {
    'ENG-1': 'Premier League', 'SPA-1': 'LaLiga', 'GER-1': 'Bundesliga',
    'ITA-1': 'Serie A', 'FRA-1': 'Ligue 1', 'POR-1': 'Liga Portugal',
    'BR-1': 'Brasileirão', 'NED-1': 'Eredivisie',
    'BEL-1': 'Jupiler Pro League', 'ARG-1': 'Liga Profesional',
    'COL-1': 'Primera A', 'PAR-1': 'Primera División — Paraguai',
    'URU-1': 'Primera División — Uruguai', 'KSA-1': 'Saudi Pro League',
    'MEX-1': 'Liga MX', 'USA-1': 'Major League Soccer',
}
SELECTION_STATES = {'played', 'in squad', 'not in squad'}
EXCLUDED_STATES = {'injured', 'absent', 'played for a different team'}
MIN_AGE, MAX_AGE = 16, 23
EWMA_ALPHA = 0.25
CUSUM_ALLOWANCE = 0.25
WATCH_WINDOW = 3
WATCH_REQUIRED_HIGH = 2
WATCH_PEER_Z_THRESHOLD = 0.75
FLAG_MIN_OBSERVATIONS = 5
FLAG_ENTRY_QUANTILE = 0.90
FLAG_RESET_QUANTILE = 0.50
FLAG_RESET_OBSERVATIONS = 3
MIN_ROUNDS_FOR_RANKING = 8
NON_USE_MIN_STREAK = 3
NON_USE_MIN_POST_SIGNAL_ROUNDS = 3
USAGE_HISTORICAL_MIN_ROUNDS = 8
USAGE_RECENT_ROUNDS = 5
USAGE_RECENCY_DAYS = 60
SEASON_LEADER_MIN_DECISIONS = 3

In [2]:
matches = pd.read_parquet(PROCESSED / 'player_match_minutes.parquet')
rosters = pd.read_parquet(RAW / 'roster_memberships.parquet')
teams = pd.read_parquet(RAW / 'teams.parquet')

state_audit = (matches.groupby('participation_state', dropna=False)
    .agg(rows=('game_id', 'size'), positive_minutes=('played', 'sum'),
         mean_minutes=('minutes_played', 'mean'))
    .sort_values('rows', ascending=False))
display(state_audit)

,rows,positive_minutes,mean_minutes
participation_state,,,
played,1146547,1146487,64.93295
not in squad,484445,0,0.00000
in squad,402788,0,0.00000
injured,174988,0,0.00000
played for a different team,104419,0,0.00000
absent,25075,0,0.00000


In [3]:
def pipe_unique(values):
    return '|'.join(sorted({str(value) for value in values.dropna()}))

def modal_value(values):
    modes = values.dropna().mode()
    return modes.iloc[0] if len(modes) else None

# Stable display identity. Position is the modal value in roster history.
identities = (rosters.sort_values('source_season_id')
    .groupby('player_id', as_index=False)
    .agg(player_name=('player_name', 'last'), player_position=('player_position', modal_value)))

team_lookup = (teams.rename(columns={
        'competition_code': 'competition_id', 'season_id': 'season_id',
        'team_id': 'club_id', 'team_name': 'club_name',
    })[['competition_id', 'season_id', 'club_id', 'club_name']]
    .assign(competition_id=lambda x: x.competition_id.astype(str),
            season_id=lambda x: x.season_id.astype('Int64'),
            club_id=lambda x: x.club_id.astype(str))
    .drop_duplicates(['competition_id', 'season_id', 'club_id']))

# Retain observed decisions; unavailability and another-club states do not become zero.
selection = matches.loc[matches['participation_state'].isin(SELECTION_STATES)].copy()
selection['club_id'] = selection['club_id'].astype(str)
selection['match_duration'] = pd.to_numeric(selection['match_duration'], errors='coerce')
selection['minutes_played'] = pd.to_numeric(selection['minutes_played'], errors='coerce').fillna(0)
selection['match_timestamp_utc'] = pd.to_datetime(selection['match_timestamp_utc'], utc=True)

ROUND_KEYS = ['competition_id', 'source_competition_id', 'season_id', 'round_id']
PLAYER_ROUND_KEYS = ['player_id', *ROUND_KEYS]
rounds = (selection.sort_values(['player_id', 'match_timestamp_utc', 'game_id'])
    .groupby(PLAYER_ROUND_KEYS, as_index=False, observed=True)
    .agg(match_date=('match_timestamp_utc', 'min'),
         club_id=('club_id', 'last'), club_ids=('club_id', pipe_unique),
         n_match_records=('game_id', 'nunique'),
         minutes_played=('minutes_played', 'sum'),
         available_minutes=('match_duration', 'sum'),
         age_decimal=('age_decimal', 'first'), age_years=('age_years', 'first'))
)
rounds['minutes_share'] = np.where(
    rounds['available_minutes'].gt(0), rounds['minutes_played'] / rounds['available_minutes'], np.nan
)
rounds['played'] = rounds['minutes_played'].gt(0)

# League-round reference: all ages before the 16–23 filter.
reference = (rounds.groupby(ROUND_KEYS, as_index=False, observed=True)
    .agg(n_players_observed=('player_id', 'nunique'),
         mean_minutes_all=('minutes_played', 'mean'),
         std_minutes_all=('minutes_played', lambda x: x.std(ddof=0)),
         share_played_all=('played', 'mean'))
)
rounds = rounds.merge(reference, on=ROUND_KEYS, how='left', validate='many_to_one')
rounds['minutes_vs_round_mean_all'] = np.where(
    rounds['mean_minutes_all'].gt(0), rounds['minutes_played'] / rounds['mean_minutes_all'], np.nan
)
rounds['round_percentile_all'] = rounds.groupby(ROUND_KEYS, observed=True)['minutes_played'].rank(
    method='average', pct=True
)

target = rounds.loc[rounds['age_years'].between(MIN_AGE, MAX_AGE, inclusive='both')].copy()
target = target.sort_values(['player_id', 'match_date', 'competition_id', 'round_id']).reset_index(drop=True)
previous_league = target.groupby('player_id', observed=True)['competition_id'].shift()
new_spell = previous_league.isna() | target['competition_id'].ne(previous_league)
target['league_spell_number'] = new_spell.astype(int).groupby(target['player_id']).cumsum()

# Descriptive ordinal encoding of involvement in each observed round.
target['response_category'] = np.select(
    [target['minutes_played'].eq(0), target['minutes_share'].le(1/3), target['minutes_share'].lt(.75)],
    [0, 1, 2], default=3
).astype('int8')

# Descriptive league-season-age benchmark; not a prospective score without backtesting.
PEER_KEYS = ['competition_id', 'source_competition_id', 'season_id', 'age_years']
peer = (target.groupby(PEER_KEYS, as_index=False, observed=True)
    .agg(peer_mean_percentile=('round_percentile_all', 'mean'),
         peer_sd_percentile=('round_percentile_all', lambda x: x.std(ddof=0)),
         peer_players=('player_id', 'nunique'))
)
target = target.merge(peer, on=PEER_KEYS, how='left', validate='many_to_one')
target['peer_residual_z'] = np.where(
    target['peer_sd_percentile'].gt(0),
    (target['round_percentile_all'] - target['peer_mean_percentile']) / target['peer_sd_percentile'],
    0.0
).clip(-4, 4)

def sequential_signals(group):
    player_id = group.name
    group = group.sort_values(['match_date', 'round_id']).copy()
    group['player_id'] = player_id
    group['career_observation_number'] = np.arange(1, len(group) + 1)
    residual = group['peer_residual_z'].fillna(0.0)
    group['ewma_z'] = residual.ewm(alpha=EWMA_ALPHA, adjust=False).mean()
    group['ewma_signal_z'] = group['ewma_z'].where(group['career_observation_number'].ge(5))
    cusum, running = [], 0.0
    for value in residual:
        running = max(0.0, running + float(value) - CUSUM_ALLOWANCE)
        cusum.append(running)
    group['cusum_positive'] = cusum
    group['rolling_minutes_5'] = group['minutes_played'].rolling(5, min_periods=1).sum()
    return group

target = (target.groupby('player_id', group_keys=False, observed=True)
    .apply(sequential_signals, include_groups=False)
    .reset_index(drop=True))
target = target.merge(identities, on='player_id', how='left', validate='many_to_one')
target = target.merge(team_lookup, on=['competition_id', 'season_id', 'club_id'],
                      how='left', validate='many_to_one')
target['league_name'] = target['competition_id'].map(LEAGUE_NAMES).fillna(target['competition_id'])
target['club_name'] = target['club_name'].fillna(target['club_id'])

In [4]:
# Club/league contexts are retained for description; the athlete EWMA is continuous.
previous_competition = target.groupby('player_id', observed=True)['competition_id'].shift()
previous_club = target.groupby('player_id', observed=True)['club_id'].shift()
new_context = (previous_competition.isna()
               | target['competition_id'].ne(previous_competition)
               | target['club_id'].ne(previous_club))
target['monitoring_context_number'] = (new_context.astype(int)
    .groupby(target['player_id']).cumsum())
monitoring_groups = target.groupby(
    ['player_id', 'monitoring_context_number'], observed=True, sort=False
)
target['monitoring_observation_number'] = monitoring_groups.cumcount() + 1
target['monitoring_ewma_z'] = target.groupby('player_id', observed=True)['peer_residual_z'].transform(
    lambda x: x.fillna(0.0).ewm(alpha=EWMA_ALPHA, adjust=False).mean()
)

flag_reference = target.loc[
    target['monitoring_observation_number'].ge(FLAG_MIN_OBSERVATIONS)
]
flag_thresholds = (flag_reference.groupby('player_position', observed=True)
    ['monitoring_ewma_z'].quantile([FLAG_RESET_QUANTILE, FLAG_ENTRY_QUANTILE]).unstack()
    .rename(columns={
        FLAG_RESET_QUANTILE: 'reset_threshold',
        FLAG_ENTRY_QUANTILE: 'entry_threshold',
    }).reset_index())
threshold_lookup = flag_thresholds.set_index('player_position').to_dict('index')

target['flag_event_number'] = 0
last_context = target.groupby('player_id', observed=True)['monitoring_context_number'].max().to_dict()
event_rows = []
for (player_id, context_number), group in monitoring_groups:
    group = group.sort_values(['match_date', 'round_id'])
    position = group['player_position'].iloc[0]
    thresholds = threshold_lookup[position]
    active_event = None
    below_reset = 0
    context_event_number = 0
    context_minutes = 0
    episode_minutes = 0
    episode_rounds = 0
    for row in group.itertuples():
        context_minutes += int(row.minutes_played)
        if active_event is None:
            episode_minutes += int(row.minutes_played)
            episode_rounds += 1
            if (episode_rounds >= FLAG_MIN_OBSERVATIONS
                    and episode_minutes > 0
                    and row.monitoring_ewma_z >= thresholds['entry_threshold']):
                context_event_number += 1
                event_rows.append({
                    'player_id': player_id,
                    'monitoring_context_number': context_number,
                    'context_event_number': context_event_number,
                    'start_index': row.Index,
                    'flag_date': row.match_date,
                    'flag_age': row.age_decimal,
                    'flag_competition_id': row.competition_id,
                    'flag_club_id': row.club_id,
                    'flag_club_name': row.club_name,
                    'episode_minutes_to_flag': episode_minutes,
                    'episode_rounds_to_flag': episode_rounds,
                    'context_minutes_to_flag': context_minutes,
                    'context_rounds_to_flag': row.monitoring_observation_number,
                    'flag_ewma_z': row.monitoring_ewma_z,
                    'entry_threshold': thresholds['entry_threshold'],
                    'close_date': pd.NaT,
                    'closure_reason': 'open',
                })
                active_event = len(event_rows) - 1
                below_reset = 0
        else:
            event = event_rows[active_event]
            below_reset = (below_reset + 1
                           if row.monitoring_ewma_z < thresholds['reset_threshold'] else 0)
            if below_reset >= FLAG_RESET_OBSERVATIONS:
                event['close_date'] = row.match_date
                event['closure_reason'] = 'signal reset'
                active_event = None
                below_reset = 0
                episode_minutes = 0
                episode_rounds = 0
    if active_event is not None and context_number != last_context[player_id]:
        event_rows[active_event]['close_date'] = group['match_date'].max()
        event_rows[active_event]['closure_reason'] = 'context changed'

flag_events = pd.DataFrame(event_rows).sort_values(
    ['player_id', 'flag_date', 'monitoring_context_number']
).reset_index(drop=True)
flag_events['player_flag_number'] = (flag_events.groupby('player_id', observed=True)
    .cumcount() + 1)
flag_events['player_flag_count'] = flag_events.groupby(
    'player_id', observed=True
)['player_id'].transform('size')
flag_events['event_open'] = flag_events['closure_reason'].eq('open')
for row in flag_events.itertuples(index=False):
    target.at[int(row.start_index), 'flag_event_number'] = int(row.player_flag_number)

# Early opportunity watches. Only the first qualifying career event is retained.
# The rule favors speed: at least two observations in a rolling window of up to
# three eligible decisions must have age-adjusted residuals >= +0.75 z.
target['watch_event_number'] = 0
watch_rows = []
for player_id, group in target.groupby('player_id', observed=True, sort=False):
    group = group.sort_values(['match_date', 'round_id'])
    career_minutes = 0
    recent_peer_z = []
    for row in group.itertuples():
        career_minutes += int(row.minutes_played)
        recent_peer_z.append(float(row.peer_residual_z) if pd.notna(row.peer_residual_z) else 0.0)
        recent_peer_z = recent_peer_z[-WATCH_WINDOW:]
        high_count = sum(value >= WATCH_PEER_Z_THRESHOLD for value in recent_peer_z)
        if high_count >= WATCH_REQUIRED_HIGH and career_minutes > 0:
            watch_rows.append({
                'player_id': player_id,
                'monitoring_context_number': row.monitoring_context_number,
                'start_index': row.Index,
                'watch_date': row.match_date,
                'watch_age': row.age_decimal,
                'watch_competition_id': row.competition_id,
                'watch_club_id': row.club_id,
                'watch_club_name': row.club_name,
                'minutes_to_watch': career_minutes,
                'rounds_to_watch': int(row.career_observation_number),
                'watch_peer_z': row.peer_residual_z,
                'watch_ewma_z': row.monitoring_ewma_z,
                'qualifying_observations': high_count,
            })
            break

watch_events = (pd.DataFrame(watch_rows)
    .sort_values(['player_id', 'watch_date', 'monitoring_context_number'])
    .drop_duplicates('player_id', keep='first')
    .reset_index(drop=True))
watch_events['player_watch_number'] = 1
watch_events['player_watch_count'] = 1
for row in watch_events.itertuples(index=False):
    target.at[int(row.start_index), 'watch_event_number'] = int(row.player_watch_number)

# One row per player for ranking and navigation.
grouped = target.groupby('player_id', observed=True)
summary = grouped.agg(
    player_name=('player_name', 'last'), player_position=('player_position', modal_value),
    first_date=('match_date', 'min'), last_date=('match_date', 'max'),
    first_age=('age_decimal', 'min'), latest_age=('age_decimal', 'last'),
    rounds=('round_id', 'size'), played_rounds=('played', 'sum'),
    total_minutes=('minutes_played', 'sum'),
    mean_round_percentile=('round_percentile_all', 'mean'),
    peak_ewma_z=('ewma_signal_z', 'max'), latest_ewma_z=('ewma_z', 'last'),
    max_cusum=('cusum_positive', 'max'), league_spells=('league_spell_number', 'max'),
    leagues=('competition_id', 'nunique')
).reset_index()

latest = (target.sort_values('match_date').groupby('player_id', as_index=False).tail(1)
    [['player_id', 'competition_id', 'league_name', 'club_id', 'club_name']])
peak_source = target.dropna(subset=['ewma_signal_z'])
peak_index = peak_source.groupby('player_id', observed=True)['ewma_signal_z'].idxmax()
peak = (target.loc[peak_index, ['player_id', 'match_date', 'age_decimal', 'competition_id',
                                'league_name', 'club_name', 'ewma_z']]
    .rename(columns={'match_date': 'signal_date', 'age_decimal': 'signal_age',
                     'competition_id': 'signal_competition_id',
                     'league_name': 'signal_league_name', 'club_name': 'signal_club_name'}))
summary = summary.merge(latest, on='player_id', how='left').merge(peak, on='player_id', how='left')
summary['play_rate'] = summary['played_rounds'] / summary['rounds']
summary['rank_eligible'] = summary['rounds'].ge(MIN_ROUNDS_FOR_RANKING) & summary['peak_ewma_z'].notna()
summary['signal_percentile'] = np.nan
mask = summary['rank_eligible']
summary.loc[mask, 'signal_percentile'] = summary.loc[mask, 'peak_ewma_z'].rank(pct=True) * 100

flag_player_summary = (flag_events.groupby('player_id', as_index=False, observed=True)
    .agg(flag_count=('player_flag_number', 'max')))
latest_flag = (flag_events.sort_values('flag_date')
    .groupby('player_id', as_index=False, observed=True).tail(1)
    [['player_id', 'flag_date', 'event_open']]
    .rename(columns={
        'flag_date': 'latest_flag_date',
        'event_open': 'latest_flag_open',
    }))
summary = (summary.merge(flag_player_summary, on='player_id', how='left', validate='one_to_one')
    .merge(latest_flag, on='player_id', how='left', validate='one_to_one'))
summary['flag_count'] = summary['flag_count'].fillna(0).astype(int)
summary['latest_flag_open'] = summary['latest_flag_open'].fillna(False).astype(bool)

watch_player_summary = (watch_events.groupby('player_id', as_index=False, observed=True)
    .agg(watch_count=('player_watch_number', 'max')))
latest_watch = (watch_events.sort_values('watch_date')
    .groupby('player_id', as_index=False, observed=True).tail(1)
    [['player_id', 'watch_date']]
    .rename(columns={'watch_date': 'latest_watch_date'}))
summary = (summary.merge(watch_player_summary, on='player_id', how='left', validate='one_to_one')
    .merge(latest_watch, on='player_id', how='left', validate='one_to_one'))
summary['watch_count'] = summary['watch_count'].fillna(0).astype(int)

# Monitoring layer: historical versus recent use across the full player trajectory.
# Transfers do not reset the sequence; unavailable states were already excluded.
data_cutoff = target['match_date'].max()
signal_date_lookup = summary.set_index('player_id')['latest_watch_date'].to_dict()
usage_rows = []
for player_id, group in target.sort_values('match_date').groupby('player_id', observed=True):
    signal_date = signal_date_lookup.get(player_id)
    post_signal = (group.loc[group['match_date'].gt(signal_date)]
                   if pd.notna(signal_date) else group.iloc[0:0])
    unused_streak = 0
    for minutes in post_signal['minutes_played'].iloc[::-1]:
        if minutes > 0:
            break
        unused_streak += 1
    used = group.loc[group['minutes_played'].gt(0), 'match_date']
    recent = group.tail(USAGE_RECENT_ROUNDS)
    historical = group.iloc[:-USAGE_RECENT_ROUNDS]
    historical_available = historical['available_minutes'].sum()
    recent_available = recent['available_minutes'].sum()
    historical_use = (historical['minutes_played'].sum() / historical_available
                      if len(historical) >= USAGE_HISTORICAL_MIN_ROUNDS
                      and historical_available > 0 else np.nan)
    recent_use = (recent['minutes_played'].sum() / recent_available
                  if len(recent) == USAGE_RECENT_ROUNDS
                  and recent_available > 0 else np.nan)
    previous_ewma = (group['monitoring_ewma_z'].iloc[-(USAGE_RECENT_ROUNDS + 1)]
                     if len(group) > USAGE_RECENT_ROUNDS else np.nan)
    current_ewma = (group['monitoring_ewma_z'].iloc[-1] if len(group) else np.nan)
    usage_rows.append({
        'player_id': player_id,
        'post_signal_rounds': len(post_signal),
        'unused_streak': unused_streak,
        'recent_5_minutes': recent['minutes_played'].sum(),
        'last_used_date': used.max() if len(used) else pd.NaT,
        'historical_usage_rounds': len(historical),
        'recent_usage_rounds': len(recent),
        'historical_usage_share': historical_use,
        'recent_usage_share': recent_use,
        'previous_ewma': previous_ewma,
        'current_ewma': current_ewma,
        'ewma_change': current_ewma - previous_ewma,
    })
usage = pd.DataFrame(usage_rows)
summary = summary.merge(usage, on='player_id', how='left', validate='one_to_one')
summary['last_observation_date'] = summary['last_date']
summary['days_since_observation'] = (
    data_cutoff - summary['last_observation_date']
).dt.total_seconds() / 86_400
summary['age_at_cutoff'] = summary['latest_age'] + (
    summary['days_since_observation'] / 365.2425
)
summary['study_cohort'] = np.where(
    summary['age_at_cutoff'].lt(MAX_AGE + 1),
    'current window',
    'completed window',
)
summary['usage_change_pp'] = 100 * (
    summary['recent_usage_share'] - summary['historical_usage_share']
)
summary['usage_plot_eligible'] = (
    summary['rank_eligible']
    & summary['watch_count'].gt(0)
    & summary['age_at_cutoff'].ge(MIN_AGE)
    & summary['age_at_cutoff'].lt(MAX_AGE + 1)
    & summary['historical_usage_rounds'].ge(USAGE_HISTORICAL_MIN_ROUNDS)
    & summary['recent_usage_rounds'].eq(USAGE_RECENT_ROUNDS)
    & summary['historical_usage_share'].notna()
    & summary['recent_usage_share'].notna()
    & summary['days_since_observation'].le(USAGE_RECENCY_DAYS)
)
summary['ewma_plot_eligible'] = (
    summary['rank_eligible']
    & summary['watch_count'].gt(0)
    & summary['age_at_cutoff'].ge(MIN_AGE)
    & summary['age_at_cutoff'].lt(MAX_AGE + 1)
    & summary['rounds'].ge(USAGE_HISTORICAL_MIN_ROUNDS)
    & summary['previous_ewma'].notna()
    & summary['current_ewma'].notna()
    & summary['days_since_observation'].le(USAGE_RECENCY_DAYS)
)
summary['non_use_flag'] = (
    summary['usage_plot_eligible']
    & summary['post_signal_rounds'].ge(NON_USE_MIN_POST_SIGNAL_ROUNDS)
    & summary['unused_streak'].ge(NON_USE_MIN_STREAK)
)
summary['non_use_watch'] = summary['non_use_flag']
summary = summary.sort_values(['signal_percentile', 'total_minutes'], ascending=False).reset_index(drop=True)

# League-season leaders: highest continuous EWMA reached during each edition.
season_keys = ['competition_id', 'season_id', 'player_id']
season_player = (target.groupby(season_keys, as_index=False, observed=True)
    .agg(season_decisions=('round_id', 'size'),
         season_minutes=('minutes_played', 'sum'),
         season_start=('match_date', 'min'),
         season_end=('match_date', 'max')))
season_peak_index = target.groupby(season_keys, observed=True)['ewma_z'].idxmax()
season_peak = (target.loc[season_peak_index, [*season_keys, 'player_name',
                    'player_position', 'club_name', 'match_date', 'age_decimal', 'ewma_z']]
    .rename(columns={'match_date': 'peak_date', 'age_decimal': 'peak_age',
                     'ewma_z': 'season_peak_ewma'}))
season_player = season_player.merge(season_peak, on=season_keys, how='inner', validate='one_to_one')
season_player = season_player.loc[
    season_player['season_decisions'].ge(SEASON_LEADER_MIN_DECISIONS)
    & season_player['season_minutes'].gt(0)
    & season_player['season_peak_ewma'].notna()
].copy()
season_context = (target.groupby(['competition_id', 'season_id'], as_index=False, observed=True)
    .agg(context_start=('match_date', 'min'), context_end=('match_date', 'max')))
season_context['spans_years'] = season_context['context_start'].dt.year.ne(
    season_context['context_end'].dt.year)
cross_year_leagues = set(season_context.groupby('competition_id', observed=True)
    ['spans_years'].mean().loc[lambda x: x.ge(.5)].index)
season_context['season_label'] = season_context.apply(
    lambda row: (f"{row.context_start.year}/{str(row.context_start.year + 1)[-2:]}"
                 if row.competition_id in cross_year_leagues
                 else str(row.context_start.year)), axis=1)
latest_season_ids = (season_context.sort_values('context_start')
    .groupby('competition_id', observed=True).tail(1)
    .set_index('competition_id')['season_id'].to_dict())
season_context['season_ongoing'] = season_context.apply(
    lambda row: (row.season_id == latest_season_ids[row.competition_id]
                 and (data_cutoff - row.context_end).days <= USAGE_RECENCY_DAYS), axis=1)
season_player = season_player.merge(season_context, on=['competition_id', 'season_id'],
                                    how='left', validate='many_to_one')
season_player['season_rank'] = (season_player.groupby(['competition_id', 'season_id'], observed=True)
    ['season_peak_ewma'].rank(method='first', ascending=False).astype(int))
season_player['season_eligible_players'] = season_player.groupby(
    ['competition_id', 'season_id'], observed=True)['player_id'].transform('size')
season_player['position_eligible_players'] = season_player.groupby(
    ['competition_id', 'season_id', 'player_position'], observed=True)['player_id'].transform('size')
season_player = (season_player.sort_values(
        ['competition_id', 'season_id', 'season_peak_ewma'],
        ascending=[True, True, False])
    .reset_index(drop=True))

audit = {
    'raw_match_rows': int(len(matches)),
    'selection_universe_rows': int(len(selection)),
    'excluded_unavailable_or_other_team_rows': int(len(matches) - len(selection)),
    'all_age_player_round_rows': int(len(rounds)),
    'target_player_round_rows': int(len(target)),
    'target_players': int(target['player_id'].nunique()),
    'players_in_multiple_leagues': int(summary['leagues'].gt(1).sum()),
    'rank_eligible_players': int(summary['rank_eligible'].sum()),
    'flag_events': int(len(flag_events)),
    'flagged_players': int(flag_events['player_id'].nunique()),
    'players_with_multiple_flags': int(flag_events.groupby('player_id').size().gt(1).sum()),
    'watch_events': int(len(watch_events)),
    'watched_players': int(watch_events['player_id'].nunique()),
    'players_with_multiple_watches': int(watch_events.groupby('player_id').size().gt(1).sum()),
    'non_use_flagged_players': int(summary['non_use_flag'].sum()),
    'usage_plot_players': int(summary['usage_plot_eligible'].sum()),
    'ewma_plot_players': int(summary['ewma_plot_eligible'].sum()),
    'season_leader_rows': int(len(season_player)),
    'league_rounds': int(len(reference)),
}
audit

{'raw_match_rows': 2338262,
 'selection_universe_rows': 2033780,
 'excluded_unavailable_or_other_team_rows': 304482,
 'all_age_player_round_rows': 2022931,
 'target_player_round_rows': 814065,
 'target_players': 18681,
 'players_in_multiple_leagues': 2465,
 'rank_eligible_players': 16016,
 'non_use_flagged_players': 965,
 'usage_plot_players': 2789,
 'league_rounds': 3994}

## **4. Methodological interpretation**

### **Statistical process monitoring**

The primary exploratory event is now an early watch. A player's first and only entry occurs when two observations in a rolling window of up to three eligible decisions have age-adjusted residuals of at least +0.75 and cumulative minutes are positive. The EWMA and watch window remain continuous across club or league changes. The legacy EWMA flag is retained only for diagnostic comparison and is not used by the website.

A prospective implementation should monitor residuals from a prospectively estimated expected-minutes model. Control limits require temporal backtesting or simulation to manage false alarms and average run length.

## **5. Analytical outputs**

The exported tables preserve round-level decisions, monitoring contexts, the first career watch event, and one player-level summary for ranking and navigation. `opportunity_watch_events.parquet` has at most one row per player.

In [5]:
# Analytical tables for subsequent stages.
target.to_parquet(PROCESSED / 'age_16_23_opportunity_signals.parquet', index=False)
summary.to_parquet(PROCESSED / 'player_opportunity_summary.parquet', index=False)
flag_events.to_parquet(PROCESSED / 'opportunity_flag_events.parquet', index=False)
watch_events.to_parquet(PROCESSED / 'opportunity_watch_events.parquet', index=False)

# Payload compacto para o index.html.
league_codes = sorted(LEAGUE_NAMES)
league_index = {code: index for index, code in enumerate(league_codes)}
club_names = sorted(target['club_name'].dropna().astype(str).unique())
club_index = {name: index for index, name in enumerate(club_names)}
histories = defaultdict(list)
for row in target.sort_values(['player_id', 'match_date']).itertuples(index=False):
    histories[str(row.player_id)].append([
        row.match_date.strftime('%Y-%m-%d'),
        round(float(row.age_decimal) * 1000),
        league_index[str(row.competition_id)],
        club_index[str(row.club_name)],
        int(row.minutes_played),
        round(float(row.minutes_share) * 1000),
        round(float(row.round_percentile_all) * 1000),
        round(float(row.peer_residual_z) * 100),
        round(float(row.ewma_z) * 100),
        round(float(row.cusum_positive) * 100),
        int(row.watch_event_number),
        round(float(row.monitoring_ewma_z) * 100),
    ])

def number(value, digits=2):
    return None if pd.isna(value) else round(float(value), digits)

def text_value(value):
    return None if pd.isna(value) else str(value)

def date_value(value):
    return None if pd.isna(value) else value.strftime('%Y-%m-%d')

players_payload = []
for row in summary.itertuples(index=False):
    players_payload.append({
        'id': str(row.player_id), 'name': row.player_name or str(row.player_id),
        'position': row.player_position or 'Unknown',
        'league': str(row.competition_id), 'club': str(row.club_name),
        'latestAge': number(row.latest_age), 'firstAge': number(row.first_age),
        'rounds': int(row.rounds), 'playedRounds': int(row.played_rounds),
        'minutes': int(row.total_minutes), 'playRate': number(row.play_rate, 3),
        'peakEwma': number(row.peak_ewma_z), 'latestEwma': number(row.latest_ewma_z),
        'maxCusum': number(row.max_cusum), 'score': number(row.signal_percentile, 1),
        'signalAge': number(row.signal_age),
        'signalDate': date_value(row.signal_date),
        'signalLeague': text_value(row.signal_competition_id),
        'signalClub': text_value(row.signal_club_name),
        'studyCohort': str(row.study_cohort),
        'flagCount': int(row.flag_count),
        'latestFlagDate': date_value(row.latest_flag_date),
        'latestFlagOpen': bool(row.latest_flag_open),
        'watchCount': int(row.watch_count),
        'latestWatchDate': date_value(row.latest_watch_date),
        'spells': int(row.league_spells), 'leagueCount': int(row.leagues),
        'ageAtCutoff': number(row.age_at_cutoff),
        'postSignalRounds': int(row.post_signal_rounds),
        'unusedStreak': int(row.unused_streak),
        'recent5Minutes': int(row.recent_5_minutes),
        'lastUsedDate': date_value(row.last_used_date),
        'lastObservedDate': date_value(row.last_observation_date),
        'nonUseFlag': bool(row.non_use_flag),
        'nonUseWatch': bool(row.non_use_watch),
        'historicalUsageRounds': int(row.historical_usage_rounds),
        'recentUsageRounds': int(row.recent_usage_rounds),
        'historicalUsage': number(row.historical_usage_share, 4),
        'recentUsage': number(row.recent_usage_share, 4),
        'usageChangePp': number(row.usage_change_pp, 1),
        'usagePlotEligible': bool(row.usage_plot_eligible),
        'previousEwma': number(row.previous_ewma, 3),
        'currentEwma': number(row.current_ewma, 3),
        'ewmaChange': number(row.ewma_change, 3),
        'ewmaPlotEligible': bool(row.ewma_plot_eligible),
        'history': histories[str(row.player_id)],
    })

season_leaders_payload = []
for row in season_player.itertuples(index=False):
    season_leaders_payload.append({
        'playerId': str(row.player_id),
        'league': str(row.competition_id),
        'seasonId': str(row.season_id),
        'seasonLabel': str(row.season_label),
        'ongoing': bool(row.season_ongoing),
        'seasonStart': date_value(row.context_start),
        'seasonEnd': date_value(row.context_end),
        'peakDate': date_value(row.peak_date),
        'peakAge': number(row.peak_age, 3),
        'peakEwma': number(row.season_peak_ewma, 3),
        'club': str(row.club_name),
        'position': str(row.player_position),
        'decisions': int(row.season_decisions),
        'minutes': int(row.season_minutes),
        'rank': int(row.season_rank),
        'eligiblePlayers': int(row.season_eligible_players),
        'positionEligiblePlayers': int(row.position_eligible_players),
    })

watches_payload = []
for row in watch_events.itertuples(index=False):
    watches_payload.append({
        'id': f'{row.player_id}:{row.player_watch_number}',
        'playerId': str(row.player_id),
        'number': int(row.player_watch_number),
        'total': int(row.player_watch_count),
        'date': date_value(row.watch_date),
        'age': number(row.watch_age, 3),
        'league': str(row.watch_competition_id),
        'club': str(row.watch_club_name),
        'minutes': int(row.minutes_to_watch),
        'rounds': int(row.rounds_to_watch),
        'peerZ': number(row.watch_peer_z, 3),
        'ewma': number(row.watch_ewma_z, 3),
        'qualifyingObservations': int(row.qualifying_observations),
    })

flags_payload = []
for row in flag_events.itertuples(index=False):
    flags_payload.append({
        'id': f'{row.player_id}:{row.player_flag_number}',
        'playerId': str(row.player_id),
        'number': int(row.player_flag_number),
        'total': int(row.player_flag_count),
        'date': date_value(row.flag_date),
        'age': number(row.flag_age, 3),
        'league': str(row.flag_competition_id),
        'club': str(row.flag_club_name),
        'minutes': int(row.episode_minutes_to_flag),
        'rounds': int(row.episode_rounds_to_flag),
        'contextMinutes': int(row.context_minutes_to_flag),
        'contextRounds': int(row.context_rounds_to_flag),
        'ewma': number(row.flag_ewma_z, 3),
        'open': bool(row.event_open),
        'closeDate': date_value(row.close_date),
        'closureReason': str(row.closure_reason),
    })

payload = {
    'meta': {
        **audit, 'minAge': MIN_AGE, 'maxAge': MAX_AGE,
        'ewmaAlpha': EWMA_ALPHA, 'cusumAllowance': CUSUM_ALLOWANCE,
        'minRoundsForRanking': MIN_ROUNDS_FOR_RANKING,
        'nonUseMinStreak': NON_USE_MIN_STREAK,
        'nonUseMinPostSignalRounds': NON_USE_MIN_POST_SIGNAL_ROUNDS,
        'usageHistoricalMinRounds': USAGE_HISTORICAL_MIN_ROUNDS,
        'usageRecentRounds': USAGE_RECENT_ROUNDS,
        'usageRecencyDays': USAGE_RECENCY_DAYS,
        'seasonLeaderMinDecisions': SEASON_LEADER_MIN_DECISIONS,
        'watchWindow': WATCH_WINDOW,
        'watchRequiredHigh': WATCH_REQUIRED_HIGH,
        'watchPeerZThreshold': WATCH_PEER_Z_THRESHOLD,
        'flagMinObservations': FLAG_MIN_OBSERVATIONS,
        'flagEntryQuantile': FLAG_ENTRY_QUANTILE,
        'flagResetQuantile': FLAG_RESET_QUANTILE,
        'flagResetObservations': FLAG_RESET_OBSERVATIONS,
        'flagThresholds': flag_thresholds.to_dict('records'),
        'dataCutoff': data_cutoff.strftime('%Y-%m-%d'),
        'generatedAt': pd.Timestamp.now(tz='UTC').isoformat(),
        'signalLabel': 'descriptive opportunity signal; not a talent probability',
    },
    'leagues': [{'id': code, 'name': LEAGUE_NAMES[code]} for code in league_codes],
    'clubs': club_names,
    'historySchema': ['date', 'age_x1000', 'league_index', 'club_index', 'minutes',
                      'share_x1000', 'round_percentile_x1000', 'peer_z_x100',
                      'ewma_z_x100', 'cusum_x100', 'watch_number',
                      'monitoring_ewma_x100'],
    'players': players_payload,
    'seasonLeaders': season_leaders_payload,
    'watches': watches_payload,
    'flags': flags_payload,
}
asset_path = ASSETS / 'player_opportunity_explorer.json'
asset_path.write_text(json.dumps(payload, ensure_ascii=False, separators=(',', ':'), allow_nan=False),
                      encoding='utf-8')
print(f'{asset_path}: {asset_path.stat().st_size / 1_000_000:.1f} MB')

/Users/gabrielcarbinatto/Documents/GitHub/statistical-player-opportunity-monitoring/assets/player_opportunity_explorer.json: 51.3 MB


In [6]:
display(summary.loc[summary['rank_eligible'], [
    'player_name', 'player_position', 'signal_age', 'signal_league_name',
    'signal_club_name', 'rounds', 'total_minutes', 'peak_ewma_z',
    'max_cusum', 'signal_percentile',
]].head(25))

,player_name,player_position,signal_age,signal_league_name,signal_club_name,rounds,total_minutes,peak_ewma_z,max_cusum,signal_percentile
0,Aaron Hickey,Defender,18.393259,Serie A,Bologna FC 1909,126,6934,4.000000,114.389676,99.996878
1,Andrea Papetti,Defender,18.007214,Serie A,Brescia Calcio (- 2025),12,917,4.000000,38.633815,99.996878
2,Rayan,Attack,19.30772,Brasileirão,Clube de Regatas Vasco da Gama,130,5603,3.995393,142.034945,99.987512
3,Tino Livramento,Defender,19.121543,Premier League,Southampton FC,131,7739,3.977163,136.597944,99.981269
4,Sulaiman Hazazi,Defender,18.116731,Saudi Pro League,Al-Wehda FC,115,3741,3.909992,62.029942,99.975025
5,Mateus Mané,Attack,18.450755,Premier League,Wolverhampton Wanderers,37,1791,3.883280,75.875607,99.968781
6,Roméo Lavia,Midfield,19.348789,Premier League,Southampton FC,100,3723,3.769444,82.138859,99.962537
7,Jair Cunha,Defender,20.244084,Brasileirão,Botafogo de Futebol e Regatas,64,1896,3.718526,41.567984,99.956294
8,Abdulmalik Al-Oyayari,Defender,20.055169,Saudi Pro League,Al-Taawoun FC,109,4322,3.708679,127.774134,99.950050
9,Wilfried Gnonto,Attack,19.288555,Premier League,Leeds United,69,2041,3.673639,57.093990,99.943806


## **6. Static visualizations**

The same visual style used in `cbf-match-report-analysis`: white background, no grid, axes only on the left and bottom, regular-weight axis and legend text, black outlines, `dpi=300`, and figures saved to `viz/` before display.

In [ ]:
import matplotlib.pyplot as plt

BLUE, RED, GOLD, INK, GRAY = '#06478f', '#b01f16', '#d97706', '#111111', '#aeb4bd'
plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['font.sans-serif'] = ['Open Sans', 'DejaVu Sans', 'Arial']

POSITION_ORDER = ['Goalkeeper', 'Defender', 'Midfield', 'Attack']


def style_axis(ax):
    ax.spines[['top', 'right']].set_visible(False)
    ax.spines[['left', 'bottom']].set_color(INK)
    ax.grid(False)
    ax.set_title(ax.get_title(), fontweight='bold')
    ax.xaxis.label.set_fontweight('normal')
    ax.yaxis.label.set_fontweight('normal')
    for label in [*ax.get_xticklabels(), *ax.get_yticklabels()]:
        label.set_fontweight('normal')
    legend = ax.get_legend()
    if legend is not None:
        for text in legend.get_texts():
            text.set_fontweight('normal')


plot_events = watch_events.merge(
    summary[['player_id', 'player_position', 'study_cohort']],
    on='player_id', how='left', validate='many_to_one',
)
plot_events['watch_age_year'] = np.floor(plot_events['watch_age']).astype(int)

In [ ]:
age_counts = (plot_events.groupby('watch_age_year', observed=True).size()
    .reindex(range(MIN_AGE, MAX_AGE + 1), fill_value=0))
x = np.arange(len(age_counts))

fig, ax = plt.subplots(figsize=(10, 5.8))
bars = ax.bar(x, age_counts.values, color=BLUE, edgecolor=INK)
for bar in bars:
    value = int(bar.get_height())
    if value:
        ax.text(bar.get_x() + bar.get_width() / 2, value + 8, f'{value:,}',
                ha='center', fontsize=8, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels([f'{age}' for age in age_counts.index])
ax.set_xlabel('Age at watch')
ax.set_ylabel('Watch events')
ax.set_title('Early opportunity watches by age')
style_axis(ax)
fig.tight_layout()
fig.savefig(VIZ_DIR / '2_01_watches_por_idade.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(9.5, 6.5))
ax.scatter(plot_events['rounds_to_watch'], plot_events['minutes_to_watch'],
           s=18, alpha=.48, color=BLUE, edgecolor=INK, linewidth=.25)
ax.set_xscale('log')
ax.set_yscale('log')
ax.set_xlabel('Decisions until watch (log scale)')
ax.set_ylabel('Minutes until watch (log scale)')
ax.set_title('Observed exposure required to produce each watch')
style_axis(ax)
fig.tight_layout()
fig.savefig(VIZ_DIR / '2_02_minutos_rodadas_ate_watch.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
position_counts = (plot_events.groupby('player_position', observed=True).size()
    .reindex(POSITION_ORDER, fill_value=0))
y = np.arange(len(position_counts))

fig, ax = plt.subplots(figsize=(9.5, 5.8))
bars = ax.barh(y, position_counts.values, color=BLUE, edgecolor=INK)
for bar in bars:
    value = int(bar.get_width())
    ax.text(value + 18, bar.get_y() + bar.get_height() / 2, f'{value:,}',
            va='center', fontsize=9, fontweight='bold')
ax.set_yticks(y)
ax.set_yticklabels(position_counts.index)
ax.set_xlabel('Watch events')
ax.set_title('Early opportunity watches by player position')
style_axis(ax)
fig.tight_layout()
fig.savefig(VIZ_DIR / '2_03_watches_por_posicao.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
player_watch_counts = watch_events.groupby('player_id', observed=True).size()
recurrence = (player_watch_counts.clip(upper=4).replace({4: '4+'})
    .value_counts().reindex([1, 2, 3, '4+'], fill_value=0))

fig, ax = plt.subplots(figsize=(8.5, 5.6))
bars = ax.bar([str(value) for value in recurrence.index], recurrence.values,
              color=[BLUE, BLUE, GOLD, RED], edgecolor=INK)
for bar, value in zip(bars, recurrence.values):
    ax.text(bar.get_x() + bar.get_width() / 2, value + 35, f'{int(value):,}',
            ha='center', fontweight='bold', fontsize=9)
ax.set_xlabel('Watches per player')
ax.set_ylabel('Players')
ax.set_title('Opportunity watches across player contexts')
style_axis(ax)
fig.tight_layout()
fig.savefig(VIZ_DIR / '2_04_recorrencia_por_jogador.png', dpi=300, bbox_inches='tight')
plt.show()

## **7. Priority references**

- Grigg & Spiegelhalter (2007), [risk-adjusted EWMA](https://doi.org/10.1198/016214506000001121).
- Steiner (1999), [time-varying EWMA limits and fast initial response](https://doi.org/10.1080/00224065.1999.11979899).
- Steiner, Cook & Farewell (2000), [risk-adjusted CUSUM](https://doi.org/10.1093/biostatistics/1.4.441).
- Adams & MacKay (2007), [Bayesian online changepoint detection](https://arxiv.org/abs/0710.3742).
- Bergkamp et al. (2019), [methodological issues in talent identification](https://doi.org/10.1007/s40279-019-01113-w).
- Mills & Bullough (2026), [early trust and professional trajectories](https://doi.org/10.1177/17479541261416476).